# Few-shot causal GPT models
Script for GPT4o-mini / GPT4 using langchain and openAI on Cloud service (Azure)

## Imports

In [0]:

import re
import os
import pandas as pd
import time # needed for openAI API rate limiting
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import KFold
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import LLMChain
from langchain_openai import AzureChatOpenAI

## Loading MAUDE data and preprocessing

In [0]:
# Data cleaning function - combined
def standardization(sent: str) -> str:
    '''
    Input: raw reviews (string)
    Output: cleaned & standardized reviews (string)
    '''
    # Convert to lowercase, remove unwanted patterns, and remove non-alphanumeric characters
    sent = re.sub(r'[^0-9a-zA-Z-ZäöüÄÖÜßéóƒÚâèåèñéçýáúåí\s]', '', sent.lower())  
    # Remove specific MAUDE patterns
    sent = re.sub(r'\(b\)\(6\)|\(b\) \(6\)|\(b\)\(4\)|\(b\) \(4\)|\[rs\]\.\\n', '', sent)
    # Replace multiple spaces and strip leading/trailing whitespaces
    sent = re.sub(r'\s+', ' ', sent).strip()  
    
    return sent

def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    '''
    Input: DataFrame with column 'text' 
    Output: Cleaned DataFrame with standardized text
    '''
    # Standardize text by applying the standardization function to each row
    df["text"] = df["text"].apply(standardization) 

    return df

# params
max_words            = 2000

## Load and preprocess dataset
filename = "data/cybersecurity_annotated_data.pq"

df = pd.read_parquet(filename)
df = clean_dataframe(df[['ID', 'text', 'label']].dropna())
df['text'] = df['text'].apply(lambda x : ' '.join(x.split(' ')[:max_words]))
df.head()


## Split data

Splitting the data in 5 chunks as used in the "traditional" models for foldwise comparability

In [0]:

X = df["text"]
y = df["label"]

# Set up 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

accuracies = []
FoldsDict = {}
for n, (train_index, test_index) in enumerate(kf.split(X)):
    
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    thisFold = {
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test
    }
    FoldsDict[n] = thisFold 
 



Selecting the fold

In [0]:
Fold = 0  #<- sets the current fold 0-4

x = FoldsDict[Fold]["X_test"]
y = FoldsDict[Fold]["y_test"] 
print(x)

## Setup the OpenAI API

In [0]:
# Initialize Header
open_api_headers = {
    "client_id": "yourID",
    "client_secret": "yourSecret"
}
                      
open_api_base = "https://chatgpt.api.YOURBASE"


# initialize the Langchain Adapter for Azure Open AI
model1 = AzureChatOpenAI(
    model="gpt-4o-mini",
    default_headers=open_api_headers,
    openai_api_type="azure",
    azure_endpoint=open_api_base,
    openai_api_key= "not_relevant",
    openai_api_version="2024-06-01"
)

## Prompt examples
Zero shot and few shot examples.

#### Zero shot _prompt_

In [0]:
PromptVersion = "ZeroShot"
sys_prompt = """Answer format:
Respond ONLY with either '0' or '1'.
Respond with '0' if the report is not cybersecurity relevant.\
Respond with '1' if the report is cybersecurity relevant.\
"""

#### Two example prompts

The Prompt name states the order in which the classes are described to the LLM. Run just one of the two


C0C1L = first the negative Class (Class 0), second the positive Class (Class 1);

C1C0L = first the positive Class (Class 1), second the negative Class (Class 0)

In [0]:
PromptVersion = "Prompt2C0C1L"
sys_prompt = """
Your role is Cybersecurity incident classification expert.
You classify whether the report describes a cybersecurity incident case based on the following instructions:

    Classify as '0' if the report contains software updates, system backup modes, or general malfunctions.
    Classify as '0' if the report describes primarily a medical issue or surgical procedure.
    Classify as '0' if the report contains information about a general system error or malfunction.
    Classify as '0' if the report does not mention cybersecurity-related concerns.
    Classify as '0' if the report relates to a defective device unrelated to cybersecurity.
    Classify as '0' if the report describes fraudulent behaviour by businesses or by companies.
    Classify as '0' if the report lists many possible root causes.

    Classify as '1' if the report mentions actions like hacking, phishing, unauthorized access, data breaches, malware, or ransomware.
    Classify as '1' if the report states unauthorized access to systems, devices, networks, or data.
    Classify as '1' if the report contains info about loss or theft of sensitive information.
    Classify as '1' if the report mentions lacking cybersecurity protection such as firewalls and antivirus software.
    Classify as '1' if the patient suspects cybersecurity attacks, even if unverified.
    Classify as '1' if the report describes fraudulent behaviour related to networks, internet or computers.
    Classify as '1' if the report mentions switched off firewalls.
    Classify as '1' if the report content appears technically not plausible.

    Do not base the classification on the word "hacking" if it relates to coughing or when describing a mechanical issue.
    Respond ONLY with '0' or '1'.
    You do not provide an explanation.
    You do not provide a summary.

"""

In [0]:
PromptVersion = "Prompt2C1C0L"
sys_prompt = """
Your role is Cybersecurity incident classification expert.
You classify whether the report describes a cybersecurity incident case based on the following instructions:

    Classify as '1' if the report mentions actions like hacking, phishing, unauthorized access, data breaches, malware, or ransomware.
    Classify as '1' if the report states unauthorized access to systems, devices, networks, or data.
    Classify as '1' if the report contains info about loss or theft of sensitive information.
    Classify as '1' if the report mentions lacking cybersecurity protection such as firewalls and antivirus software.
    Classify as '1' if the patient suspects cybersecurity attacks, even if unverified.
    Classify as '1' if the report describes fraudulent behaviour related to networks, internet or computers.
    Classify as '1' if the report mentions switched off firewalls.
    Classify as '1' if the report content appears technically not plausible.

    Classify as '0' if the report contains software updates, system backup modes, or general malfunctions.
    Classify as '0' if the report describes primarily a medical issue or surgical procedure.
    Classify as '0' if the report contains information about a general system error or malfunction.
    Classify as '0' if the report does not mention cybersecurity-related concerns.
    Classify as '0' if the report relates to a defective device unrelated to cybersecurity.
    Classify as '0' if the report describes fraudulent behaviour by businesses or by companies.
    Classify as '0' if the report lists many possible root causes.

    Do not base the classification on the word "hacking" if it relates to coughing or when describing a mechanical issue.
    Respond ONLY with '0' or '1'.
    You do not provide an explanation.
    You do not provide a summary.

"""

#### Human Prompt

In [0]:

human_prompt =  "Please classify the following report into *relevant* for cybersecurity or *not relevant* for cybersecurity: "

## Classifying...
**Note:** Make sure that the desired sys_prompt is used. It is the systemprompt cell that was run last   


In [0]:
#++++++++++++++  This sets the prompt cell that was run last as system prompt
used_sys_prompt = sys_prompt


RES_DF = pd.DataFrame()

prompt_template = ChatPromptTemplate([
    ("system", used_sys_prompt),
    ("user", human_prompt + """ {report} """ ) ])
                                     
# Create the LLMChain for classification
classifier_chain = LLMChain(llm=model1, prompt=prompt_template)

for n, text_to_classify in enumerate(x):

    time.sleep(5) #limiter for the openAI API
    print(f"+++++++++++++++++ this is number {n}" )
    try:
        classification = classifier_chain.run({"report": text_to_classify })
    except:
        classification = 3 #if a content filter kicks in, eg violence
        print(f"Conten Filter blocked: {text_to_classify}")

    if len(str(classification)) == 3: #the prompt "respond with" leads to answers as string "'0'" or integer "0"  
        classification = str(classification)[1]
    elif len(str(classification)) > 3: #in case the reposne is anything else the a number or a number in ''s
        classification = 4
    try : 
        classification = int(classification)
    except ValueError:
        classification = 2
    
    
    print(f"Classification after: {classification}")
    print(f"label: {y.iloc[n]}")
    RES_DF = RES_DF._append({"report": text_to_classify, "label":y.iloc[n], "predict": classification },ignore_index=True)
#save to excel

RES_DF.to_excel( PromptVersion + "Fold_" + str(Fold) + ".xlsx", index = False)


## Evaluate the results

In [0]:

def eval_model(y_test,y_pred):
    # value counts of the predicted labels
    print(RES_DF["predict"].value_counts())
  
    # Evaluate the model
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred, digits= 3))

In [0]:

eval_model(RES_DF["label"].values ,RES_DF["predict"].values)